# ATML PA3 - Task 1: Large Language Model Fine-Tuning
## Kaggle Execution Notebook

This notebook runs the complete end-to-end pipeline for **Task 1** of EE-5102/CS-6304 (ATML PA3):
1. **MetaMathQA Filtering & Activation Probe Sampling** (seed 6304, GSM8K test exclusion, max 2048 tokens)
2. **Shared Baseline Evaluation** (`Qwen/Qwen3-0.6B` zero-shot on GSM8K, HellaSwag, BoolQ, ARC-Easy)
3. **Five Training Variants**:
   - Full Fine-Tuning (Full FT, $\lambda=0$)
   - Full FT + L2-SP ($\lambda=10^{-3}$)
   - LoRA-4 ($r=4, \alpha=8$)
   - LoRA-16 ($r=16, \alpha=32$)
   - OPLoRA-16 ($r=16, \alpha=32, k=128$, with frozen singular bases $U_k, V_k$)
4. **Zero-Shot Benchmark Evaluation** across all 7 conditions via `lm-eval` Python API
5. **Weight-Space & Activation Diagnostics** on `down_proj` in layers 0, 13, and 27
6. **Qualitative Evaluation & 42-Response Annotation** (`first_plot_questions.yaml`)
7. **Continuous Checkpoint Archiving & Downloadable Zip Packaging**

### Step 0: Hardware Verification & Repository Setup

In [ ]:
# Verify GPU
!nvidia-smi

import os
import sys

# Prevent CUDA memory fragmentation on 15GB/16GB GPUs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO_URL = "https://github.com/ahmad-sarfraz345/ATML-PA3.git"
REPO_DIR = "/kaggle/working/ATML-PA3"

if os.path.exists("scripts/run_task1.py"):
    print("Already inside ATML-PA3 repository directory.")
    !git fetch origin && git reset --hard origin/main
elif os.path.exists(REPO_DIR):
    print("Updating existing repository in /kaggle/working/ATML-PA3...")
    %cd {REPO_DIR}
    !git fetch origin && git reset --hard origin/main
else:
    print("Cloning repository into /kaggle/working/ATML-PA3...")
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())
print("Working directory:", os.getcwd())

### Step 1: Install Environment Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
print("Dependencies successfully installed.")

### Step 2: Modular Stage Controls & Archiving Helper
Any stage or checkpoint that is already completed on disk is **automatically cached and skipped** in 1 second.

In [ ]:
import shutil
from pathlib import Path

def auto_package_outputs(tag="latest"):
    """Helper to archive outputs/task1 continuously so progress is never lost."""
    source_dir = "outputs/task1"
    zip_filename = f"/kaggle/working/task1_outputs_{tag}"
    if os.path.exists(source_dir) and any(Path(source_dir).iterdir()):
        shutil.make_archive(zip_filename, 'zip', source_dir)
        size_mb = Path(zip_filename + '.zip').stat().st_size / (1024 * 1024)
        print(f"[Archive] Saved {zip_filename}.zip ({size_mb:.2f} MB)")

# Stage execution toggles
STAGE_PREPARE_DATA       = True
STAGE_EVAL_BASELINE      = True   # Automatically loads cached baseline_results.json if already done
STAGE_TRAIN_FULL_FT      = True   # Memory-optimized FP32 master weights with active logits slice
STAGE_TRAIN_FULL_FT_L2SP = True   # Memory-optimized in-place L2-SP proximity gradient
STAGE_TRAIN_LORA_4       = True   # Automatically skips if checkpoint_step_1250.pt exists
STAGE_TRAIN_LORA_16      = True   # Automatically skips if checkpoint_step_1250.pt exists
STAGE_TRAIN_OPLORA_16    = True   # Automatically skips if checkpoint_step_1250.pt exists
STAGE_EVAL_BENCHMARKS    = True   # Zero-shot evaluation across available checkpoints
STAGE_DIAGNOSTICS        = True   # Weight-space and activation diagnostics
STAGE_QUALITATIVE        = True   # 168 qualitative generations & 42-item annotations
STAGE_ANALYSIS           = True   # Summary tables, learning curves, and resource usage

MODEL_ID = "Qwen/Qwen3-0.6B"
SEED = 6304

### Step 3: Prepare MetaMathQA 40k & Sample Activation Probe Set

In [ ]:
if STAGE_PREPARE_DATA:
    print("--- Executing Stage: Prepare Data ---")
    !python -u -m scripts.run_task1 --stage prepare_data --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_outputs("data")

### Step 4: Evaluate Baseline (Unchanged Model, 0 Updates)

In [ ]:
if STAGE_EVAL_BASELINE:
    print("--- Executing Stage: Evaluate Baseline ---")
    !python -u -m scripts.run_task1 --stage eval_baseline --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_outputs("baseline")

### Step 5: Train Model Variants
1. Full Fine-Tuning ($lr=2\times 10^{-5}, \lambda=0$)
2. Full FT + L2-SP ($lr=2\times 10^{-5}, \lambda=10^{-3}$)
3. LoRA Rank 4 ($r=4, \alpha=8, lr=2\times 10^{-4}$)
4. LoRA Rank 16 ($r=16, \alpha=32, lr=2\times 10^{-4}$)
5. OPLoRA Rank 16 ($r=16, \alpha=32, k=128, lr=2\times 10^{-4}$)

In [ ]:
if STAGE_TRAIN_FULL_FT:
    print("--- Executing Stage: Train Full FT ---")
    !python -u -m scripts.run_task1 --stage train_full_ft --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_outputs("full_ft")

if STAGE_TRAIN_FULL_FT_L2SP:
    print("--- Executing Stage: Train Full FT + L2-SP ---")
    !python -u -m scripts.run_task1 --stage train_full_ft_l2sp --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_outputs("full_ft_l2sp")

if STAGE_TRAIN_LORA_4:
    print("--- Executing Stage: Train LoRA-4 ---")
    !python -u -m scripts.run_task1 --stage train_lora_4 --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_outputs("lora_4")

if STAGE_TRAIN_LORA_16:
    print("--- Executing Stage: Train LoRA-16 ---")
    !python -u -m scripts.run_task1 --stage train_lora_16 --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_outputs("lora_16")

if STAGE_TRAIN_OPLORA_16:
    print("--- Executing Stage: Train OPLoRA-16 ---")
    !python -u -m scripts.run_task1 --stage train_oplora_16 --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_outputs("training_all")

### Step 6: Zero-Shot Benchmark Evaluation (7 Conditions x 4 Tasks)

In [ ]:
if STAGE_EVAL_BENCHMARKS:
    print("--- Executing Stage: Evaluate Checkpoints on Benchmarks ---")
    !python -u -m scripts.run_task1 --stage eval_benchmarks --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_outputs("benchmarks")

### Step 7: Weight-Space & Activation Diagnostics

In [ ]:
if STAGE_DIAGNOSTICS:
    print("--- Executing Stage: Diagnostics ---")
    !python -u -m scripts.run_task1 --stage diagnostics --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_outputs("diagnostics")

### Step 8: Qualitative Generations (168 Total, 42 Annotated)

In [ ]:
if STAGE_QUALITATIVE:
    print("--- Executing Stage: Qualitative Generations ---")
    !python -u -m scripts.run_task1 --stage qualitative --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_outputs("qualitative")

### Step 9: Synthesize Results, Figures, and Tables

In [ ]:
if STAGE_ANALYSIS:
    print("--- Executing Stage: Analysis & Plots ---")
    !python -u -m scripts.run_task1 --stage analysis --model_name_or_path {MODEL_ID} --seed {SEED}

from IPython.display import Image, display, Markdown
import pandas as pd

# Display Learning Curves
if os.path.exists("outputs/task1/learning_curves.png"):
    display(Image("outputs/task1/learning_curves.png"))

# Display Benchmark Summary Table
if os.path.exists("outputs/task1/benchmark_summary.md"):
    with open("outputs/task1/benchmark_summary.md", "r") as f:
        display(Markdown(f.read()))

# Display Resource Usage Summary
if os.path.exists("outputs/task1/resource_usage_summary.md"):
    with open("outputs/task1/resource_usage_summary.md", "r") as f:
        display(Markdown(f.read()))

### Step 10: Final Output Package & Download Links

In [ ]:
zip_filename = "/kaggle/working/task1_outputs"
source_dir = "outputs/task1"

os.makedirs(source_dir, exist_ok=True)
files = list(Path(source_dir).rglob("*"))
file_count = len([f for f in files if f.is_file()])

if file_count > 0:
    print(f"Archiving final {file_count} files from {source_dir} to {zip_filename}.zip ...")
    shutil.make_archive(zip_filename, 'zip', source_dir)
    zip_size_mb = Path(zip_filename + '.zip').stat().st_size / (1024 * 1024)
    print(f"SUCCESS: Complete Task 1 outputs packaged at {zip_filename}.zip ({zip_size_mb:.2f} MB)")
    print("You can download task1_outputs.zip directly from the Kaggle Output pane on the right.")
else:
    print(f"Warning: No files found in {source_dir} to package.")